## MLflow's Model Registry

In [5]:
from mlflow.tracking import MlflowClient


MLFLOW_TRACKING_URI = "http://mlflow.ml.svc.cluster.local"

### Interacting with the MLflow tracking server

The `MlflowClient` object allows us to interact with...
- an MLflow Tracking Server that creates and manages experiments and runs.
- an MLflow Registry Server that creates and manages registered models and model versions. 

To instantiate it we need to pass a tracking URI and/or a registry URI

In [6]:
client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

for exp in client.search_experiments():
    print(f"experiment_id: {exp.experiment_id}, name: {exp.name}")

experiment_id: 1, name: nyc-taxi-experiment
experiment_id: 0, name: Default


In [7]:
client.create_experiment(name="my-cool-experiment")

'2'

In [9]:
for exp in client.search_experiments():
    print(f"experiment_id: {exp.experiment_id}, name: {exp.name}")

experiment_id: 2, name: my-cool-experiment
experiment_id: 1, name: nyc-taxi-experiment
experiment_id: 0, name: Default


Let's check the latest versions for the experiment with id `1`...

In [19]:
from mlflow.entities import ViewType

runs = client.search_runs(
    experiment_ids=["1"],
    filter_string="metrics.rmse < 7",
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=5,
    order_by=["metrics.rmse ASC"]
)


In [20]:
for run in runs:
    print(f"run id: {run.info.run_id}, rmse: {run.data.metrics['rmse']:.4f}")

run id: 8dc53ce8003744f793bb7354175dc837, rmse: 6.2714
run id: 89b57f9a4aa0491bb76317c8bd5bf292, rmse: 6.2714
run id: c135ae7257eb49d4a20d7647db10471c, rmse: 6.2714
run id: 8c9f1d24623e4b859d34fdc4bda9ea81, rmse: 6.2754
run id: e8a0e1f78a2a4bf287f77cada27ce002, rmse: 6.2816


### Interacting with the Model Registry

In this section We will use the `MlflowClient` instance to:

1. Register a new version for the experiment `nyc-taxi-regressor`
2. Retrieve the latests versions of the model `nyc-taxi-regressor` and check that a new version `4` was created.
3. Transition the version `4` to "Staging" and adding annotations to it.

In [21]:
import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [22]:
run_id = "2bd0142bd24749fc9b408b75deb4cf9f"
model_uri = f"runs:/{run_id}/model"
mlflow.register_model(model_uri=model_uri, name="nyc-taxi-regressor")

Successfully registered model 'nyc-taxi-regressor'.
2026/03/11 09:25:47 WARNING mlflow.tracking._model_registry.fluent: Run with id 2bd0142bd24749fc9b408b75deb4cf9f has no artifacts at artifact path 'model', registering model based on models:/m-bd6f45220eec42e19bff9f37172aaf91 instead
2026/03/11 09:25:47 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: nyc-taxi-regressor, version 1
Created version '1' of model 'nyc-taxi-regressor'.


<ModelVersion: aliases=[], creation_timestamp=1773221147628, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1773221147628, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='2bd0142bd24749fc9b408b75deb4cf9f', run_link='', source='models:/m-bd6f45220eec42e19bff9f37172aaf91', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>

In [25]:
registered_models = client.search_registered_models()

for rm in registered_models:
    print(f"name: {rm.name}, latest_versions: {rm.latest_versions}")

name: nyc-taxi-regressor, latest_versions: [<ModelVersion: aliases=[], creation_timestamp=1773221327820, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1773221327820, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='c9c6db67f5594eb7b0eabfccb47adf8b', run_link='', source='models:/m-43330206232648efb121f98e54380a64', status='READY', status_message=None, tags={}, user_id='', version='2', workspace='default'>]


In [41]:
model_name = "nyc-taxi-regressor"
#old: latest_versions = client.get_latest_versions(name=model_name)
model_versions = client.search_model_versions(f"name='{model_name}'")

for mv in model_versions:
    full_mv = client.get_model_version(name=model_name, version=mv.version)
    print(f"version: {full_mv.version}, aliases: {full_mv.aliases}, tags: {full_mv.tags}")

version: 2, aliases: ['Staging'], tags: {}
version: 1, aliases: ['Production'], tags: {}


In [37]:
client.set_registered_model_alias(name=model_name, alias="Staging", version=2)
client.set_registered_model_alias(name=model_name, alias="Production", version=1) # old version

In [42]:
# let's check
model_versions = client.search_model_versions(f"name='{model_name}'")

for mv in model_versions:
    full_mv = client.get_model_version(name=model_name, version=mv.version)
    print(f"version: {full_mv.version}, aliases: {full_mv.aliases}, tags: {full_mv.tags}")

version: 2, aliases: ['Staging'], tags: {}
version: 1, aliases: ['Production'], tags: {}


In [ ]:
# Update the description with the new alias and date (this does not change the
# version number)
from datetime import datetime

new_stage = "Staging"
date = datetime.today().date()
client.update_model_version(
    name=model_name,
    version=2,
    description=f"The model version 2 was transitioned to {new_stage} on {date}"
)

<ModelVersion: aliases=['Staging'], creation_timestamp=1773221327820, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='The model version 2 was transitioned to Staging on 2026-03-11', last_updated_timestamp=1773222423099, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='c9c6db67f5594eb7b0eabfccb47adf8b', run_link='', source='models:/m-43330206232648efb121f98e54380a64', status='READY', status_message=None, tags={}, user_id='', version='2', workspace='default'>

### Comparing versions and selecting the new "Production" model

In the last section, we will retrieve models registered in the model registry and compare their performance on an unseen test set. The idea is to simulate the scenario in which a deployment engineer has to interact with the model registry to decide whether to update the model version that is in production or not.

These are the steps:

1. Load the test dataset, which corresponds to the NYC Green Taxi data from the month of March 2021.
2. Download the `DictVectorizer` that was fitted using the training data and saved to MLflow as an artifact, and load it with pickle.
3. Preprocess the test set using the `DictVectorizer` so we can properly feed the regressors.
4. Make predictions on the test set using the model versions that are currently in the "Staging" and "Production" stages, and compare their performance.
5. Based on the results, update the "Production" model version accordingly.


**Note: the model registry doesn't actually deploy the model to production when you transition a model to the "Production" stage, it just assign a label to that model version. You should complement the registry with some CI/CD code that does the actual deployment.**

In [57]:
from sklearn.metrics import root_mean_squared_error
import pandas as pd


def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df


def preprocess(df, dv):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    train_dicts = df[categorical + numerical].to_dict(orient='records')
    return dv.transform(train_dicts)


def test_model(name, alias, X_test, y_test):
    model = mlflow.pyfunc.load_model(f"models:/{name}@{alias}")
    y_pred = model.predict(X_test)
    return {"rmse": root_mean_squared_error(y_test, y_pred)}

In [46]:
!wget "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-03.parquet" -O mlops-zoomcamp/01-intro/data/green_tripdata_2021-03.parquet

--2026-03-11 10:35:28--  https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-03.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 65.9.23.179, 65.9.23.94, 65.9.23.7, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|65.9.23.179|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1474538 (1.4M) [binary/octet-stream]
Saving to: ‘mlops-zoomcamp/01-intro/data/green_tripdata_2021-03.parquet’

mlops-zoomcamp/01-i 100%[===================>]   1.41M  --.-KB/s    in 0.05s   

2026-03-11 10:35:29 (28.2 MB/s) - ‘mlops-zoomcamp/01-intro/data/green_tripdata_2021-03.parquet’ saved [1474538/1474538]



In [50]:
df = read_dataframe("~/mlops-zoomcamp/01-intro/data/green_tripdata_2021-03.parquet")

In [51]:
client.download_artifacts(run_id=run_id, path='preprocessor', dst_path='.')

'/home/jovyan/preprocessor'

In [52]:
import pickle

with open("preprocessor/preprocessor.b", "rb") as f_in:
    dv = pickle.load(f_in)

In [53]:
X_test = preprocess(df, dv)

In [54]:
target = "duration"
y_test = df[target].values

In [58]:
%time test_model(name=model_name, alias="Production", X_test=X_test, y_test=y_test)

CPU times: user 402 ms, sys: 124 ms, total: 526 ms
Wall time: 1.13 s


{'rmse': 6.659623830022514}

In [61]:
%time test_model(name=model_name, alias="Staging", X_test=X_test, y_test=y_test)

CPU times: user 14.5 s, sys: 22.1 s, total: 36.6 s
Wall time: 5min 15s


{'rmse': 6.877612674697775}

In [ ]:
# won't do this because staging is worse than production
client.transition_model_version_stage(
    name=model_name,
    version=4,
    stage="Production",
    archive_existing_versions=True
)

<ModelVersion: creation_timestamp=1652971637398, current_stage='Production', description='The model version 4 was transitioned to Staging on 2022-05-19', last_updated_timestamp=1652972763255, name='nyc-taxi-regressor', run_id='b8904012c84343b5bf8ee72aa8f0f402', run_link=None, source='./mlruns/1/b8904012c84343b5bf8ee72aa8f0f402/artifacts/model', status='READY', status_message=None, tags={}, user_id=None, version=4>